# INTERPRETER Subagent Step-by-Step Test

이 노트북은 Deep Interpreter의 **INTERPRETER subagent**를 각 Phase별로 직접 테스트할 수 있도록 구성되어 있습니다.

## Workflow Overview

```
Phase 0: RAG Preparation (Optional)
    └── prepare_rag_literature() → rag_mode 결정

Phase 1: Load Context
    └── read papers.json, strategic_plan.md, load_analysis_data()

Phase 2: Generate Interpretation
    └── (Manual) LLM 해석 생성

Phase 3: Claim Validation
    └── validate_claim_fulltext() or validate_claim_literature()

Phase 4: Quality Check
    └── validate_quality() → score ≥ 0.7?

Phase 5: Write Final
    └── write /partN_final.md
```

---
## Setup

In [1]:
import sys
import os
import json
from pathlib import Path

# Add project root to path
PROJECT_ROOT = Path(os.getcwd()).parent
sys.path.insert(0, str(PROJECT_ROOT))

# Load environment variables
from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

print(f"Project Root: {PROJECT_ROOT}")
print(f"Python Path: {sys.path[0]}")

Project Root: /Users/jungj2/Dropbox/Metadata/aiagent/DeepMAST
Python Path: /Users/jungj2/Dropbox/Metadata/aiagent/DeepMAST


In [2]:
# Import INTERPRETER tools
from src.agents.deep_interpreter.tools import (
    # Data Loader
    load_analysis_data,
    get_figure_path,
    ANALYSIS_FILES,
    # Literature
    search_literature,
    format_references,
    # Quality
    validate_quality,
    validate_claim_literature,
    get_improvement_suggestions,
    get_claim_validation_summary,
    QUALITY_CRITERIA,
    # RAG
    prepare_rag_literature,
    search_literature_fulltext,
    validate_claim_fulltext,
    check_rag_availability_sync,
    get_rag_status,
    is_rag_mode_active,
    # Report
    generate_html_report,
    # Think
    think_tool,
)

print("✅ All tools imported successfully")

✅ All tools imported successfully


In [3]:
# Configuration
DATA_DIR = PROJECT_ROOT / "res"
PAPERS_DIR = DATA_DIR / "papers"

# File paths
PART1_PAPERS_PATH = DATA_DIR / "part1_papers.json"
LIANA_PAPERS_PATH = DATA_DIR / "liana_papers.json"
STRATEGIC_PLAN_PATH = DATA_DIR / "strategic_plan.md"
DATA_PATH_FILE = DATA_DIR / "data_path.txt"

print(f"Data Directory: {DATA_DIR}")
print(f"\nFiles exist:")
print(f"  part1_papers.json: {PART1_PAPERS_PATH.exists()}")
print(f"  liana_papers.json: {LIANA_PAPERS_PATH.exists()}")
print(f"  strategic_plan.md: {STRATEGIC_PLAN_PATH.exists()}")

Data Directory: /Users/jungj2/Dropbox/Metadata/aiagent/DeepMAST/res

Files exist:
  part1_papers.json: True
  liana_papers.json: True
  strategic_plan.md: True


---
## Phase 0: RAG Preparation (Optional)

RAG 모드는 Docker 인프라가 필요합니다:
- `langconnect-rag` 서버 (http://localhost:8080)
- `pubmed-download` MCP 서버 (http://localhost:8001/mcp)

RAG가 없으면 Abstract 모드로 fallback됩니다.

In [4]:
# Step 0-1: Check RAG availability
print("=" * 60)
print("Phase 0-1: Checking RAG Availability")
print("=" * 60)

rag_available = check_rag_availability_sync()
print(f"\nRAG Available: {rag_available}")

if not rag_available:
    print("\n⚠️ RAG not available. Will use Abstract mode for validation.")
    print("   To enable RAG, start Docker containers:")
    print("   - langconnect-rag on http://localhost:8000")
    print("   - pubmed-download on http://localhost:8002/mcp")

Phase 0-1: Checking RAG Availability
[DEBUG] check_rag_availability: using stdio transport
[DEBUG]   CCI_RAG_PYTHON=/Users/jungj2/Dropbox/Metadata/mcp/langconnect_mcp/.venv/bin/python
[DEBUG]   CCI_RAG_SERVER=/Users/jungj2/Dropbox/Metadata/mcp/langconnect_mcp/mcpserver/mcp_server.py
[DEBUG]   CCI_RAG_BASE_URL=http://localhost:8888
[DEBUG] check_rag_availability: got 11 tools
[DEBUG] check_rag_availability: calling health tool
[DEBUG] check_rag_availability: health check result = True

RAG Available: True


In [5]:
PART1_PAPERS_PATH

PosixPath('/Users/jungj2/Dropbox/Metadata/aiagent/DeepMAST/res/part1_papers.json')

In [ ]:
# Step 0-2: Prepare RAG for Part 1 (if available)
print("=" * 60)
print("Phase 0-2: Prepare RAG Literature (Part 1)")
print("=" * 60)

# This will download full-text papers and index them
# Takes ~30 seconds per 10 papers

rag_result = prepare_rag_literature(
    papers_json_path=str(PART1_PAPERS_PATH),
    part_name="part1"
)

from src.agents.deep_interpreter.tools.literature_rag import prepare_rag_literature_sync
rag_result = prepare_rag_literature_sync("res/part1_papers.json", "part1")
print(rag_result)  # Should show rag_mode: True     

print(f"\nRAG Preparation Result:")
print(json.dumps(rag_result, indent=2))

# Store mode for later use
RAG_MODE = rag_result.get("rag_mode", False)
print(f"\n{'✅ RAG Mode Active' if RAG_MODE else '⚠️ Using Abstract Mode'}")

Phase 0-2: Prepare RAG Literature (Part 1)
[DEBUG] prepare_rag_literature_sync CALLED
[DEBUG] papers_json_path: res/part1_papers.json
[DEBUG] part_name: part1
[DEBUG] Event loop running: True
[DEBUG] prepare_rag_for_interpretation START
[DEBUG] Checking RAG availability...
[DEBUG] check_rag_availability: using cached result = True
[DEBUG] RAG available: True
[DEBUG] Loading papers from: res/part1_papers.json
[DEBUG] ERROR: Papers file not found: res/part1_papers.json
[DEBUG] prepare_rag_literature_sync RESULT:
[DEBUG]   rag_mode: False
[DEBUG]   papers_count: 0
[DEBUG]   fulltext_count: 0
[DEBUG]   message: Papers file not found: res/part1_papers.json
{'rag_mode': False, 'collection_id': None, 'papers_count': 0, 'fulltext_count': 0, 'message': 'Papers file not found: res/part1_papers.json'}


In [13]:
RAG_MODE=True
# Step 0-3: Check downloaded papers (if RAG mode)
if RAG_MODE:
    print("=" * 60)
    print("Phase 0-3: Check Downloaded Papers")
    print("=" * 60)
    
    papers_dir = DATA_DIR / "papers"
    if papers_dir.exists():
        files = list(papers_dir.glob("pmid_*.txt"))
        print(f"\nDownloaded papers: {len(files)}")
        for f in files[:5]:
            size = f.stat().st_size / 1024
            print(f"  {f.name}: {size:.1f} KB")
        if len(files) > 5:
            print(f"  ... and {len(files) - 5} more")
else:
    print("Skipping - RAG mode not active")

Phase 0-3: Check Downloaded Papers

Downloaded papers: 0


---
## Phase 1: Load Context

INTERPRETER는 다음 파일들을 읽어야 합니다:
1. `/partN_papers.json` - 유일한 citation source
2. `/strategic_plan.md` - 가설 확인
3. `/data_path.txt` - 실제 데이터 경로
4. `load_analysis_data()` - 분석 결과 및 figure 경로

In [ ]:
# Step 1-1: Load Papers JSON
print("=" * 60)
print("Phase 1-1: Load Papers JSON")
print("=" * 60)

with open(PART1_PAPERS_PATH) as f:
    part1_papers_data = json.load(f)

papers = part1_papers_data.get("papers", [])
print(f"\nLoaded {len(papers)} papers from part1_papers.json")
print(f"\nFirst 3 papers:")
for i, paper in enumerate(papers[:3]):
    print(f"\n  [{i+1}] PMID: {paper.get('pmid')}")
    print(f"      Title: {paper.get('title', '')[:60]}...")
    print(f"      Search Type: {paper.get('search_type')}")

In [ ]:
# Step 1-2: Load Strategic Plan
print("=" * 60)
print("Phase 1-2: Load Strategic Plan")
print("=" * 60)

if STRATEGIC_PLAN_PATH.exists():
    with open(STRATEGIC_PLAN_PATH) as f:
        strategic_plan = f.read()
    print(f"\nStrategic Plan (first 500 chars):")
    print(strategic_plan[:500])
    print("...")
else:
    print("⚠️ strategic_plan.md not found")
    strategic_plan = None

In [ ]:
# Step 1-3: Load Analysis Data
print("=" * 60)
print("Phase 1-3: Load Analysis Data (Tool Call)")
print("=" * 60)

# This is the actual tool call
analysis_data = load_analysis_data(data_path=str(DATA_DIR))

print(f"\nAnalysis Data Keys: {list(analysis_data.keys())}")

# Show figures
if "figures" in analysis_data:
    print(f"\nFigures:")
    for name, path in list(analysis_data["figures"].items())[:5]:
        print(f"  {name}: {path}")

# Show search context
if "spatial_search_context" in analysis_data:
    ctx = analysis_data["spatial_search_context"]
    print(f"\nSpatial Search Context:")
    print(f"  Pathways: {len(ctx.get('pathways', []))}")
    print(f"  Cell Types: {len(ctx.get('celltypes', []))}")

---
## Phase 2: Generate Interpretation

실제로는 LLM이 해석을 생성합니다. 여기서는 샘플 해석을 사용하여 Phase 3, 4를 테스트합니다.

In [ ]:
# Step 2-1: Sample Interpretation (for testing)
print("=" * 60)
print("Phase 2-1: Sample Interpretation")
print("=" * 60)

# Sample interpretation with PMID citations
SAMPLE_INTERPRETATION = """
### Analysis 1-1: Spatial Clustering

#### Figure Description
Spatial visualization of cell clusters across the tissue section reveals distinct 
regional organization with tumor cells predominantly localized in clusters 4, 16, and 176.

#### Quantitative Findings
- Cluster 4 contains 35.2% of all cells and shows high CDH1 expression (p < 0.001)
- Cluster 16 demonstrates 2.5-fold enrichment for proliferation markers
- Neighborhood enrichment z-score between tumor and stromal clusters: 3.8

#### Biological Interpretation
The spatial organization of tumor cells in clusters 4 and 16 suggests a highly 
organized tumor microenvironment with distinct functional domains. The enrichment 
of CDH1 in cluster 4 indicates epithelial characteristics consistent with 
E-cadherin's role in maintaining epithelial integrity (PMID: 38341519).

The spatial proximity of tumor-associated macrophages (TAMs) to tumor cell clusters 
suggests active immune-tumor crosstalk. M2-like TAMs are known to promote tumor 
drug resistance through various mechanisms including cytokine secretion and 
immunosuppression (PMID: 38341519).

Interestingly, the gradient of pathway activity from cluster 4 to cluster 176 
**(Novel Finding)** has not been previously reported in similar breast cancer contexts.
"""

print(SAMPLE_INTERPRETATION)

In [ ]:
# Step 2-2: (Optional) Search Full-Text for Context (RAG Mode)
print("=" * 60)
print("Phase 2-2: Search Full-Text for Context")
print("=" * 60)

if RAG_MODE:
    # Search for supporting context from full-text papers
    search_results = search_literature_fulltext(
        query="tumor-associated macrophages drug resistance breast cancer",
        part_name="part1",
        limit=3
    )
    
    print(f"\nSearch Results: {len(search_results)} chunks found")
    for i, result in enumerate(search_results):
        print(f"\n  [{i+1}] Score: {result.get('score', 'N/A')}")
        content = result.get('page_content', '')[:200]
        print(f"      Content: {content}...")
else:
    print("⚠️ RAG Mode not active - skipping full-text search")
    print("   Use paper abstracts from part1_papers.json instead")

---
## Phase 3: Claim Validation

각 PMID 인용에 대해 claim이 실제 논문 내용과 일치하는지 검증합니다.

- **RAG Mode**: `validate_claim_fulltext()` - 전문 기반 시맨틱 검색
- **Abstract Mode**: `validate_claim_literature()` - Abstract 기반 키워드 매칭

In [ ]:
# Step 3-1: Extract claims from interpretation
print("=" * 60)
print("Phase 3-1: Extract Claims with PMID")
print("=" * 60)

import re

# Find sentences with PMID citations
pmid_pattern = r'([^.]*PMID[:\s]*\d+[^.]*)'
claims_with_pmid = re.findall(pmid_pattern, SAMPLE_INTERPRETATION)

# Also extract the PMID numbers
pmid_numbers = re.findall(r'PMID[:\s]*(\d+)', SAMPLE_INTERPRETATION)

print(f"\nFound {len(claims_with_pmid)} claims with PMID citations:")
for i, claim in enumerate(claims_with_pmid):
    print(f"\n  [{i+1}] {claim.strip()[:100]}...")
    
print(f"\nPMID numbers: {pmid_numbers}")

In [ ]:
# Step 3-2: Get paper abstract for validation
print("=" * 60)
print("Phase 3-2: Get Paper Abstract")
print("=" * 60)

# Find paper with PMID 38341519 from our papers
target_pmid = "38341519"
paper_abstract = None

for paper in papers:
    if paper.get("pmid") == target_pmid:
        paper_abstract = paper.get("abstract", "")
        print(f"\nFound paper:")
        print(f"  PMID: {paper.get('pmid')}")
        print(f"  Title: {paper.get('title')}")
        print(f"\n  Abstract (first 300 chars):")
        print(f"  {paper_abstract[:300]}...")
        break

if not paper_abstract:
    print(f"⚠️ Paper with PMID {target_pmid} not found in papers.json")
    # Use first available paper for demo
    if papers:
        paper_abstract = papers[0].get("abstract", "")
        target_pmid = papers[0].get("pmid")
        print(f"   Using first paper instead: PMID {target_pmid}")

In [ ]:
# Step 3-3A: Validate Claim (Abstract Mode)
print("=" * 60)
print("Phase 3-3A: Validate Claim - Abstract Mode (Tool Call)")
print("=" * 60)

# Sample claim to validate
test_claim = "M2-like TAMs promote tumor drug resistance through cytokine secretion and immunosuppression"

print(f"\nClaim: {test_claim}")
print(f"PMID: {target_pmid}")

# Validate using Abstract mode
validation_result = validate_claim_literature(
    claim=test_claim,
    pmid=target_pmid,
    paper_abstract=paper_abstract
)

print(f"\n📊 Validation Result:")
print(json.dumps(validation_result, indent=2))

# Interpret result
alignment = validation_result.get("alignment")
recommendation = validation_result.get("recommendation")
print(f"\n🔍 Interpretation:")
print(f"   Alignment: {alignment}")
print(f"   Recommendation: {recommendation}")

if alignment == "strong":
    print("   ✅ Keep citation as-is")
elif alignment == "weak":
    print("   ⚠️ Consider strengthening with additional reference")
else:
    print("   ❌ Revise claim or mark as (Novel Finding)")

In [ ]:
# Step 3-3B: Validate Claim (RAG Mode) - if available
print("=" * 60)
print("Phase 3-3B: Validate Claim - RAG Mode (Tool Call)")
print("=" * 60)

if RAG_MODE:
    # Validate using full-text RAG
    rag_validation = validate_claim_fulltext(
        claim=test_claim,
        pmid=target_pmid,
        part_name="part1"
    )
    
    print(f"\n📊 RAG Validation Result:")
    print(json.dumps(rag_validation, indent=2))
    
    # Compare with Abstract mode
    print(f"\n📈 Comparison:")
    print(f"   Abstract Mode Confidence: {validation_result.get('confidence', 'N/A')}")
    print(f"   RAG Mode Confidence: {rag_validation.get('confidence', 'N/A')}")
else:
    print("⚠️ RAG Mode not active - skipping full-text validation")

In [ ]:
# Step 3-4: Validate multiple claims
print("=" * 60)
print("Phase 3-4: Validate Multiple Claims")
print("=" * 60)

# List of claims to validate
test_claims = [
    {
        "claim": "CDH1 indicates epithelial characteristics in breast cancer",
        "pmid": target_pmid
    },
    {
        "claim": "Tumor-associated macrophages promote drug resistance",
        "pmid": target_pmid
    },
    {
        "claim": "Spatial transcriptomics reveals cell type heterogeneity",
        "pmid": target_pmid
    }
]

validation_results = []

for item in test_claims:
    result = validate_claim_literature(
        claim=item["claim"],
        pmid=item["pmid"],
        paper_abstract=paper_abstract
    )
    validation_results.append(result)
    print(f"\n  Claim: {item['claim'][:50]}...")
    print(f"  → Alignment: {result['alignment']}, Confidence: {result['confidence']}")

# Get summary
summary = get_claim_validation_summary(validation_results)
print(f"\n📊 Validation Summary:")
print(json.dumps(summary, indent=2))

---
## Phase 4: Quality Check

해석의 품질을 4가지 기준으로 평가합니다:
- **statistics**: 정량적 통계 (p-값, fold change 등)
- **references**: 문헌 인용 (PMID)
- **mechanism**: 생물학적 메커니즘 설명
- **specificity**: 구체적 표현 (모호한 표현 회피)

In [ ]:
# Step 4-1: Check Quality Criteria
print("=" * 60)
print("Phase 4-1: Quality Criteria")
print("=" * 60)

print("\nQuality Criteria:")
for name, criteria in QUALITY_CRITERIA.items():
    print(f"\n  {name} (weight: {criteria['weight']})")
    print(f"    {criteria['description']}")

In [ ]:
# Step 4-2: Validate Quality (Tool Call)
print("=" * 60)
print("Phase 4-2: Validate Quality (Tool Call)")
print("=" * 60)

# Prepare references list
references = [
    {"pmid": "38341519", "title": "Targeting M2-like tumor-associated macrophages..."},
    {"pmid": "39834406", "title": "The role of cell cycle-related genes..."},
]

# Validate quality
quality_result = validate_quality(
    interpretation=SAMPLE_INTERPRETATION,
    references=references,
    use_llm=False  # Set to True to use reflection-tool MCP
)

print(f"\n📊 Quality Validation Result:")
print(json.dumps(quality_result, indent=2))

# Interpret result
score = quality_result.get("score", 0)
passed = quality_result.get("passed", False)

print(f"\n🎯 Quality Score: {score:.3f}")
print(f"   Threshold: 0.7")
print(f"   Status: {'✅ PASSED' if passed else '❌ FAILED'}")

if not passed:
    print(f"\n📝 Feedback: {quality_result.get('feedback', '')}")

In [ ]:
# Step 4-3: Get Improvement Suggestions
print("=" * 60)
print("Phase 4-3: Improvement Suggestions")
print("=" * 60)

suggestions = get_improvement_suggestions(quality_result)

if suggestions:
    print("\n🔧 Suggestions to improve quality:")
    for i, suggestion in enumerate(suggestions, 1):
        print(f"   {i}. {suggestion}")
else:
    print("\n✅ No improvements needed - all criteria met!")

In [ ]:
# Step 4-4: Quality Loop Simulation
print("=" * 60)
print("Phase 4-4: Quality Loop Simulation")
print("=" * 60)

MAX_ITERATIONS = 3
current_interpretation = SAMPLE_INTERPRETATION
iteration = 0

while iteration < MAX_ITERATIONS:
    iteration += 1
    print(f"\n--- Iteration {iteration} ---")
    
    result = validate_quality(
        interpretation=current_interpretation,
        references=references,
        use_llm=False
    )
    
    score = result.get("score", 0)
    print(f"Score: {score:.3f}")
    
    if result.get("passed", False):
        print(f"✅ Quality threshold met at iteration {iteration}")
        break
    else:
        print(f"❌ Score < 0.7, need improvement")
        if iteration < MAX_ITERATIONS:
            print(f"   Would improve and retry...")
            # In real scenario, LLM would improve the interpretation here
        else:
            print(f"   Max iterations reached, proceeding with current quality")

print(f"\n📊 Final Score: {score:.3f}")

In [ ]:
# Step 4-5: (Optional) Think Tool for Reflection
print("=" * 60)
print("Phase 4-5: Think Tool for Reflection")
print("=" * 60)

try:
    reflection_text = f"""
    Reflecting on interpretation quality:
    
    Current Score: {score:.3f}
    Statistics Score: {quality_result.get('statistics_score', 'N/A')}
    References Score: {quality_result.get('references_score', 'N/A')}
    Mechanism Score: {quality_result.get('mechanism_score', 'N/A')}
    Specificity Score: {quality_result.get('specificity_score', 'N/A')}
    
    What areas need improvement?
    """
    
    think_result = think_tool(reflection=reflection_text)
    print(f"\n🧠 Think Tool Result:")
    print(think_result)
except Exception as e:
    print(f"⚠️ Think tool not available: {e}")
    print("   (Requires reflection-tool MCP server)")

---
## Phase 5: Write Final

최종 해석을 `/partN_final.md` 파일로 저장합니다.

In [ ]:
# Step 5-1: Prepare Final Content
print("=" * 60)
print("Phase 5-1: Prepare Final Content")
print("=" * 60)

# Build final markdown content
final_content = f"""# Part 1: Spatial & Pathway Analysis - Test Output

{SAMPLE_INTERPRETATION}

---

## Part 1 Summary

### Key Findings
1. Tumor cells show distinct spatial organization in clusters 4, 16, and 176
2. CDH1 enrichment indicates epithelial characteristics (PMID: 38341519)
3. Pathway activity gradient from cluster 4 to 176 **(Novel Finding)**

### Hypothesis Status
- H1 (Spatial Organization): **Supported** - Clear regional organization observed

### Quality Metrics
- Statistics included: 3
- Literature citations: 2 PMIDs
- Novel findings: 1
- Quality score: {score:.2f}

### Validation Notes
All PMID citations validated:
- PMID 38341519: {validation_result.get('alignment', 'N/A')} support (confidence: {validation_result.get('confidence', 'N/A')})
"""

print("Final content prepared:")
print(f"  Length: {len(final_content)} characters")
print(f"  Lines: {len(final_content.splitlines())}")

In [ ]:
# Step 5-2: Write to File
print("=" * 60)
print("Phase 5-2: Write Final File")
print("=" * 60)

output_path = DATA_DIR / "part1_final_test.md"

with open(output_path, "w") as f:
    f.write(final_content)

print(f"\n💾 Written to: {output_path}")
print(f"   File size: {output_path.stat().st_size} bytes")

In [ ]:
# Step 5-3: Verify Written Content
print("=" * 60)
print("Phase 5-3: Verify Written Content")
print("=" * 60)

with open(output_path) as f:
    written_content = f.read()

print(f"\n📄 First 500 characters of written file:")
print(written_content[:500])
print("...")

---
## Summary

이 노트북에서 테스트한 INTERPRETER 워크플로우:

In [ ]:
# Summary
print("=" * 60)
print("INTERPRETER Workflow Summary")
print("=" * 60)

summary = {
    "Phase 0 - RAG Preparation": {
        "tool": "prepare_rag_literature()",
        "rag_mode": RAG_MODE,
        "status": "✅ Completed" if RAG_MODE else "⚠️ Fallback to Abstract"
    },
    "Phase 1 - Load Context": {
        "tool": "load_analysis_data()",
        "papers_loaded": len(papers),
        "status": "✅ Completed"
    },
    "Phase 2 - Generate Interpretation": {
        "method": "LLM Generation (sample used)",
        "status": "✅ Completed"
    },
    "Phase 3 - Claim Validation": {
        "tool": "validate_claim_fulltext()" if RAG_MODE else "validate_claim_literature()",
        "claims_validated": len(test_claims),
        "status": "✅ Completed"
    },
    "Phase 4 - Quality Check": {
        "tool": "validate_quality()",
        "final_score": f"{score:.3f}",
        "passed": "✅ Yes" if score >= 0.7 else "❌ No",
        "status": "✅ Completed"
    },
    "Phase 5 - Write Final": {
        "output": str(output_path),
        "status": "✅ Completed"
    }
}

print(json.dumps(summary, indent=2, default=str))

In [ ]:
# Cleanup test file
print("\n🧹 Cleanup:")
if output_path.exists():
    # output_path.unlink()  # Uncomment to delete test file
    print(f"   Test file kept at: {output_path}")
    print("   (Uncomment unlink() to auto-delete)")